In [32]:
import os
import io
import requests
import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
from supabase import create_client
import onnxruntime as ort

# 1. Inisialisasi Supabase
load_dotenv()
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(url, key)

# 2. Ambil data dari Supabase
response = supabase.table("pothole_detections").select("*").execute()
df_db = pd.DataFrame(response.data)
print(f"Berhasil mengambil {len(df_db)} baris data dari Supabase.")

# 3. Muat Model ONNX
session = ort.InferenceSession("model_anda.onnx")
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# 4. Fungsi Preprocessing Gambar dari URL ke NumPy Array
def preprocess_image_from_url(image_url, target_size=(224, 224)):
    # Unduh gambar
    resp = requests.get(image_url)
    img = Image.open(io.BytesIO(resp.content)).convert('RGB')

    # Resize gambar
    img = img.resize(target_size)

    # Konversi ke NumPy & normalisasi (0.0 - 1.0)
    img_np = np.array(img).astype(np.float32) / 255.0

    # Ubah format dari (Height, Width, Channels) ke (Channels, Height, Width) -> HWC ke CHW
    img_np = np.transpose(img_np, (2, 0, 1))

    # Tambahkan dimensi Batch: (3, 224, 224) -> (1, 3, 224, 224)
    img_np = np.expand_dims(img_np, axis=0)

    return img_np

# 5. Jalankan Prediksi untuk Setiap Gambar di DataFrame
# Asumsi nama kolom di Supabase adalah 'image_url'
prediksi_list = []

for index, row in df_db.iterrows():
    try:
        input_tensor = preprocess_image_from_url(row['image_url'])
        outputs = session.run([output_name], {input_name: input_tensor})
        hasil = outputs[0]

        # Simpan hasil (misal: kelas terprediksi)
        prediksi_list.append(np.argmax(hasil))
    except Exception as e:
        print(f"Error memproses baris {index}: {e}")
        prediksi_list.append(None)

# Masukkan hasil prediksi kembali ke DataFrame
df_db['prediksi_onnx'] = prediksi_list
print(df_db[['id', 'image_url', 'prediksi_onnx']].head())

Berhasil mengambil 263 baris data dari Supabase.


NoSuchFile: [ONNXRuntimeError] : 3 : NO_SUCHFILE : Load model from model_anda.onnx failed:Load model model_anda.onnx failed. File doesn't exist